# Part 1: Exploring the Game Data

In [1]:
%load_ext autoreload
%autoreload 2

### Imports

In [2]:
import sys
from nba_api.stats.endpoints import leaguegamefinder
import duckdb

## Opening the Data

In [3]:
finder = leaguegamefinder.LeagueGameFinder(
    league_id_nullable="00",
    season_type_nullable="Regular Season",
    date_from_nullable="10/20/2026",
    date_to_nullable="10/20/2026",
)

In [4]:
df = finder.get_data_frames()[0]

In [5]:
df.shape

(6, 28)

In [6]:
df.columns.tolist()

['SEASON_ID',
 'TEAM_ID',
 'TEAM_ABBREVIATION',
 'TEAM_NAME',
 'GAME_ID',
 'GAME_DATE',
 'MATCHUP',
 'WL',
 'MIN',
 'PTS',
 'FGM',
 'FGA',
 'FG_PCT',
 'FG3M',
 'FG3A',
 'FG3_PCT',
 'FTM',
 'FTA',
 'FT_PCT',
 'OREB',
 'DREB',
 'REB',
 'AST',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PLUS_MINUS']

In [7]:
df[["GAME_ID","SEASON_ID", "GAME_DATE", "TEAM_ABBREVIATION", "MATCHUP", "WL", "PTS"]]

,GAME_ID,SEASON_ID,GAME_DATE,TEAM_ABBREVIATION,MATCHUP,WL,PTS
0,0022600003,22026,2026-10-20,SAS,SAS vs. OKC,L,104
1,0022600003,22026,2026-10-20,OKC,OKC @ SAS,W,116
2,0022600001,22026,2026-10-20,DET,DET vs. BOS,W,121
3,0022600001,22026,2026-10-20,BOS,BOS @ DET,L,113
4,0022600002,22026,2026-10-20,PHI,PHI @ NYK,W,106
5,0022600002,22026,2026-10-20,NYK,NYK vs. PHI,L,100


In [8]:
df.head(4)

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22026,1610612759,SAS,San Antonio Spurs,0022600003,2026-10-20,SAS vs. OKC,L,239,104,...,0.880,10,35,45,21,6,11,13,24,-12.0
1,22026,1610612760,OKC,Oklahoma City Thunder,0022600003,2026-10-20,OKC @ SAS,W,241,116,...,0.800,10,37,47,24,8,10,11,22,12.0
2,22026,1610612765,DET,Detroit Pistons,0022600001,2026-10-20,DET vs. BOS,W,239,121,...,0.875,14,35,49,27,7,5,17,22,8.0
3,22026,1610612738,BOS,Boston Celtics,0022600001,2026-10-20,BOS @ DET,L,240,113,...,0.762,12,27,39,28,10,5,14,26,-8.0


In [9]:
df.dtypes

SEASON_ID                str
TEAM_ID                int64
TEAM_ABBREVIATION        str
TEAM_NAME                str
GAME_ID                  str
GAME_DATE                str
MATCHUP                  str
WL                       str
MIN                    int64
PTS                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PLUS_MINUS           float64
dtype: object

First Check results:
- 6 rows from first three games due to 2 teams a game.
- No home and away columns which could leave behind vital information, which can be extracted from the matchup string data.
- No tracked W-L record or last 5 games W-L, which could also be useful.
- Game date is stored as a string rather than as a date
- Team_id stored as an integer instead of a string

### Table Creation

In [10]:
con = duckdb.connect("../data/nba.duckdb")

In [11]:
con.execute("""
            CREATE TABLE IF NOT EXISTS raw_team_games(
                SEASON_ID   VARCHAR,
                TEAM_ID     VARCHAR,
                TEAM_ABBREVIATION VARCHAR,
                TEAM_NAME   VARCHAR,
                GAME_ID     VARCHAR,
                GAME_DATE   DATE,
                MATCHUP     VARCHAR,
                WL          VARCHAR,
                MIN         INTEGER,
                PTS         INTEGER,
                FGM         INTEGER,
                FGA         INTEGER,
                FG_PCT      DOUBLE,
                FG3M        INTEGER,
                FG3A        INTEGER,
                FG3_PCT     DOUBLE,
                FTM         INTEGER,
                FTA         INTEGER,
                FT_PCT      DOUBLE,
                OREB        INTEGER,
                DREB        INTEGER,
                REB         INTEGER,
                AST         INTEGER,
                STL         INTEGER,
                BLK         INTEGER,
                TOV         INTEGER,
                PF          INTEGER,
                PLUS_MINUS  INTEGER,
                INGESTED_AT TIMESTAMP,
                PRIMARY KEY (GAME_ID, TEAM_ID)
            )
            """)

In [12]:
con.sql("DESCRIBE raw_team_games")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ SEASON_ID         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ TEAM_ID           │ VARCHAR     │ NO      │ PRI     │ NULL    │ NULL    │
│ TEAM_ABBREVIATION │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ TEAM_NAME         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ GAME_ID           │ VARCHAR     │ NO      │ PRI     │ NULL    │ NULL    │
│ GAME_DATE         │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ MATCHUP           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WL                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ MIN               │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ PTS       

In [13]:
con.execute("""
            CREATE TABLE IF NOT EXISTS players (
                player_id   VARCHAR,
                name    VARCHAR,
                height_cm   INTEGER,
                fg_pct  DOUBLE,
                birth_date  DATE,
                loaded_at   TIMESTAMP,
                PRIMARY KEY (player_id)
            );
            """)

In [14]:
con.sql("DESCRIBE players")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ player_id   │ VARCHAR     │ NO      │ PRI     │ NULL    │ NULL    │
│ name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ height_cm   │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ fg_pct      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ birth_date  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ loaded_at   │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

### Games first test

In [16]:
from nba_predictor_v2.ingest.games import fetch_games, create_table, load_games


In [17]:
test = fetch_games("10/20/2026")
test.shape

(6, 29)

In [18]:
con = duckdb.connect("../data/nba.duckdb")
create_table(con)

In [21]:
df = fetch_games("10/20/2026")
load_games(con, df)
con.sql("SELECT COUNT(*) FROM raw_team_games")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            6 │
└──────────────┘